# Solution B1 · A tree on the Titanic: overfitting, tuning and readable rules

**Type**: Implement with scikit-learn

## Objective

Build a pipeline with a classification tree on the Titanic data (imputation, encoding of the categoricals with one-hot or ordinal), measure how much a tree **without limits** overfits, choose `max_depth`, `min_samples_leaf` and `ccp_alpha` with `GridSearchCV`
and read the rules of the final tree.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier, export_text

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
NUM = ["pclass", "age", "siblings_spouses", "parents_children", "fare"]
CAT = ["sex", "port", "deck", "title"]

## Commented solution

In [ ]:
def build(encoding, **tree_params):
    enc = OneHotEncoder(handle_unknown="ignore") if encoding == "ohe" else OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    prep = ColumnTransformer([("num", SimpleImputer(strategy="median"), NUM),
                              ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), enc), CAT)], sparse_threshold=0)
    return make_pipeline(prep, DecisionTreeClassifier(random_state=0, **tree_params))

In [ ]:
configs = {"no limits": {}, "max_depth=3": {"max_depth": 3}, "max_depth=5": {"max_depth": 5},
           "min_samples_leaf=10": {"min_samples_leaf": 10}, "min_samples_leaf=20": {"min_samples_leaf": 20}, "ccp_alpha=0.005": {"ccp_alpha": 0.005}}
cv_auc = pd.DataFrame(index=list(configs), columns=["ohe", "ord"], dtype=float)
for name, params in configs.items():
    for encoding in ["ohe", "ord"]:
        cv_auc.loc[name, encoding] = cross_val_score(build(encoding, **params), X_tr, y_tr, cv=cv, scoring="roc_auc").mean()
cv_auc.round(3)

In [ ]:
grid = {"decisiontreeclassifier__max_depth": [2, 3, 4, 5, 6, 8, None], "decisiontreeclassifier__min_samples_leaf": [1, 5, 10, 20, 40],
        "decisiontreeclassifier__ccp_alpha": [0, 0.002, 0.005, 0.01]}
search = GridSearchCV(build("ohe"), grid, cv=cv, scoring="roc_auc").fit(X_tr, y_tr)
best = {k.split("__")[1]: v for k, v in search.best_params_.items()}
test_auc = roc_auc_score(y_te, search.predict_proba(X_te)[:, 1])
n_leaves = search.best_estimator_[-1].get_n_leaves()
print(f"best parameters: {best} | CV AUC {search.best_score_:.4f} | test AUC {test_auc:.4f} | leaves {n_leaves}")

In [ ]:
model = search.best_estimator_
names = list(model[:-1].get_feature_names_out())
root = names[model[-1].tree_.feature[0]]
rules = export_text(model[-1], feature_names=names)
print("feature at the root:", root)
print("\n".join(rules.splitlines()[:14]))

## Check

In [ ]:
with verify():
    check(cv_auc.notna().all().all() and cv_auc.loc["no limits"].max() < 0.76, "the tree without limits has AUC below 0.76 (overfitting)", "DecisionTreeClassifier(random_state=0) with no parameters")
    check(cv_auc.loc["max_depth=3"].min() > 0.83 and cv_auc.drop("no limits").min().min() > 0.8, "with any regularization the AUC goes above 0.8 (over 0.83 with max_depth=3)", "compare the rows")
    check(abs(cv_auc.loc["max_depth=3", "ohe"] - cv_auc.loc["max_depth=3", "ord"]) < 0.02, "the encoding (ohe or ordinal) changes little: less than 0.02 with max_depth=3", "compare the ohe and ord columns")
    check(set(best) == {"max_depth", "min_samples_leaf", "ccp_alpha"} and best["max_depth"] in (3, 4, 5) and test_auc > 0.83, "the search picks a depth between 3 and 5; test AUC above 0.83", "keys without the prefix 'decisiontreeclassifier__'")
    check(3 <= n_leaves <= 40, "the final tree is small: from 3 to 40 leaves", "get_n_leaves() of the last step")
    check(isinstance(root, str) and ("sex" in root or "title" in root) and "class:" in rules and rules.count("\n") >= 6, "the root is `sex` or `title`; the rules are a text with the classes", "export_text(tree, feature_names=names)")
done()

## What you learned

- **A tree without limits memorizes:** cross-validation AUC 0.72 against 0.85 with `max_depth=3`. It is not a flaw of the algorithm, it is its nature: without a stopping criterion it builds a leaf for each group of examples. The regularization parameters (`max_depth`, `min_samples_leaf`, `ccp_alpha`)
  are not optional.
- **The best tree is small:** the search picks depth 3, with no other constraints, with test AUC 0.86 and **8 leaves**. With very few parameters a tree gets performance close to much more complex models, and it is readable.
- **The encoding of the categoricals (one-hot or ordinal) matters little** with `max_depth=3`: the tree chooses thresholds and the one-hot makes them explicit. With many levels the one-hot fragments the data and the ordinal encoding (with a sensible order) is more efficient.
- **The rules are the product**: the root is the most informative question (`sex`/`title`: "women and children first"), and in three or four levels you read the rules that explain almost the whole prediction. It is the reason a single tree still makes sense when you need to explain.

## Common pitfall

Reporting the rules of a tree as "the true structure of the phenomenon". With slightly different data the root and the lower levels would change (C2). The rules should be read as a **description** of this model, not as a law.

## Interview question

<details><summary>A decision tree is interpretable: how far do you trust its rules?</summary>

The rules describe precisely how that model decides, but a single tree is unstable: small changes in the data can change the root and the thresholds. That is why I regularize it (depth or minimum leaves chosen in cross-validation), check the stability with a bootstrap and use the rules as
an explanation of the model, not as a discovery about the world. If accuracy is needed I use a forest or a boosting and explain with permutation importances and SHAP.

</details>